# EDA REFIT

## 1. Các trường dữ liệu trong dataset

Mục tiêu: kiểm kê **tên cột thực tế**, ý nghĩa, đơn vị và sự hiện diện theo house. Bước này chỉ đọc header, không tính thống kê dòng hay biến đổi dữ liệu.

In [1]:
import csv
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASET = "REFIT"
DESCRIPTIONS = {
    "Time": ("Thời điểm đo dạng lịch; đã hiệu chỉnh giờ mùa hè UK", "datetime"),
    "Unix": ("Thời điểm đo dạng Unix timestamp", "giây UTC"),
    "Aggregate": ("Công suất tác dụng toàn nhà", "W"),
    "Issues": ("Cờ tổng công suất IAM vượt Aggregate", "0/1"),
    **{f"Appliance{i}": (f"Công suất cổng IAM {i}; thiết bị cụ thể tùy house", "W") for i in range(1, 10)},
}


root = Path.cwd()
if not (root / "data/cleaned").exists():
    root = root.parent.parent
base = root / "data/cleaned" / DATASET
present = defaultdict(set)
files = defaultdict(int)
for path in base.rglob("*.csv"):
    if path.name == "appliances_metadata.csv":
        group = "metadata thiết bị"
    elif "Environmental_data" in path.parts:
        group = "môi trường"
    else:
        group = "điện"
    house = next((part for part in path.parts if part.startswith("House_")), path.stem.replace("CLEAN_", ""))
    with path.open(encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))
    files[group] += 1
    for field in header:
        present[(group, field)].add(house)
rows = []
for (group, field), houses in sorted(present.items()):
    meaning, unit = DESCRIPTIONS.get(field, (f"Công suất tác dụng của thiết bị {field}" if group == "điện" else "Cần đối chiếu tài liệu nguồn", "W" if group == "điện" else ""))
    rows.append({"nhóm": group, "trường gốc": field, "ý nghĩa": meaning, "đơn vị": unit, "số house có trường": len(houses), "house": ", ".join(sorted(houses))})
dictionary = pd.DataFrame(rows)
print("Số file CSV theo nhóm:", dict(files))
display(dictionary.drop(columns="house"))

Số file CSV theo nhóm: {'điện': 20}


,nhóm,trường gốc,ý nghĩa,đơn vị,số house có trường
0,điện,Aggregate,Công suất tác dụng toàn nhà,W,20
1,điện,Appliance1,Công suất cổng IAM 1; thiết bị cụ thể tùy house,W,20
2,điện,Appliance2,Công suất cổng IAM 2; thiết bị cụ thể tùy house,W,20
3,điện,Appliance3,Công suất cổng IAM 3; thiết bị cụ thể tùy house,W,20
4,điện,Appliance4,Công suất cổng IAM 4; thiết bị cụ thể tùy house,W,20
5,điện,Appliance5,Công suất cổng IAM 5; thiết bị cụ thể tùy house,W,20
6,điện,Appliance6,Công suất cổng IAM 6; thiết bị cụ thể tùy house,W,20
7,điện,Appliance7,Công suất cổng IAM 7; thiết bị cụ thể tùy house,W,20
8,điện,Appliance8,Công suất cổng IAM 8; thiết bị cụ thể tùy house,W,20
9,điện,Appliance9,Công suất cổng IAM 9; thiết bị cụ thể tùy house,W,20


### Ghi chú diễn giải

- Cả 20 file house đều có 13 cột giống nhau.
- `Appliance1`–`Appliance9` là **vị trí cảm biến**, không phải 9 loại thiết bị chung cho mọi house. Muốn gọi tên thiết bị phải dùng bảng ánh xạ trong `data/raw/REFIT/CLEAN_READ_ME_081116.txt`.
- Các giá trị công suất là công suất tác dụng (W); nhịp đo danh định 8 giây.
- `Issues` là cờ chất lượng, không phải công suất thiết bị.

Nguồn mô tả: `data/raw/REFIT/CLEAN_READ_ME_081116.txt`.